# 08 — Matrix export and static LCA

**Audience:** Users who want file-based Premise matrices or want to calculate outside a Brightway project database.

**Prerequisites:** `premise[bw25]`, `bw_processing`, `bw2calc`, a configured Brightway source database, and a valid IAM key.

**Learning goals:** export A/B matrices, build a `bw_processing` datapackage, calculate a static LCIA score, and reproduce the solve directly with SciPy.


## Outline

1. Export one scenario to matrix CSV files.
2. Read matrix indices and values.
3. Build technosphere, biosphere, and characterization resources.
4. Calculate and verify a score.


In [ ]:
import csv
import os
from pathlib import Path

import bw2calc as bc
import bw2data as bd
import bw_processing as bwp
import numpy as np

from premise import NewDatabase

PROJECT = "ecoinvent-3.12-cutoff"
SOURCE_DATABASE = "ecoinvent-3.12-cutoff"
BIOSPHERE_DATABASE = "ecoinvent-3.12-biosphere"
PREMISE_KEY = os.environ.get("PREMISE_KEY")
MATRIX_ROOT = Path("export/tutorial-matrices")
SCENARIO = {"model": "remind", "pathway": "SSP2-NDC", "year": 2030}

if not PREMISE_KEY:
    raise RuntimeError("Set PREMISE_KEY before running this tutorial.")
bd.projects.set_current(PROJECT)


## 1. Export matrices

Passing a root directory creates `<root>/<model>/<pathway>/<year>/` beneath it.


In [ ]:
ndb = NewDatabase(
    scenarios=[SCENARIO.copy()],
    source_db=SOURCE_DATABASE,
    source_version="3.12",
    biosphere_name=BIOSPHERE_DATABASE,
    key=PREMISE_KEY,
)
ndb.update(["electricity"])
ndb.write_db_to_matrices(filepath=str(MATRIX_ROOT))

MATRIX_DIR = (
    MATRIX_ROOT / SCENARIO["model"] / SCENARIO["pathway"] / str(SCENARIO["year"])
)


## 2. Read indices and exchange arrays

Premise uses semicolon-delimited CSV files. Index labels remain available for selecting activities and biosphere flows.


In [ ]:
def read_indices_csv(file_path: Path) -> dict:
    indices = {}
    with file_path.open(encoding="utf-8") as handle:
        rows = csv.reader(handle, delimiter=";")
        next(rows, None)
        for row in rows:
            indices[tuple(str(value) for value in row[:4])] = int(row[4])
    return indices


def read_matrix_csv(file_path: Path):
    array = np.genfromtxt(file_path, delimiter=";", skip_header=1)
    indices = np.array(
        list(zip(array[:, 1].astype(int), array[:, 0].astype(int))),
        dtype=bwp.INDICES_DTYPE,
    )
    values = array[:, 2]
    flips = array[:, -1].astype(bool)
    return indices, values, flips


A_indices = read_indices_csv(MATRIX_DIR / "A_matrix_index.csv")
B_indices = read_indices_csv(MATRIX_DIR / "B_matrix_index.csv")


## 3. Build a static datapackage

This minimal method characterizes fossil carbon dioxide with a factor of one. Replace it with a full LCIA method for research results.


In [ ]:
datapackage = bwp.create_datapackage()

indices, values, flips = read_matrix_csv(MATRIX_DIR / "A_matrix.csv")
datapackage.add_persistent_vector(
    matrix="technosphere_matrix",
    indices_array=indices,
    data_array=values,
    flip_array=flips,
)

indices, values, _ = read_matrix_csv(MATRIX_DIR / "B_matrix.csv")
datapackage.add_persistent_vector(
    matrix="biosphere_matrix",
    indices_array=indices,
    data_array=values,
    flip_array=None,
)

characterization_ids = [
    identifier
    for label, identifier in B_indices.items()
    if "carbon dioxide, fossil" in label[0].lower()
]
characterization_indices = np.array(
    [(identifier, identifier) for identifier in characterization_ids],
    dtype=bwp.INDICES_DTYPE,
)
datapackage.add_persistent_vector(
    matrix="characterization_matrix",
    indices_array=characterization_indices,
    data_array=np.ones(len(characterization_indices)),
)


## 4. Calculate a static score

Select a unique activity label in production code; the short substring search here keeps the tutorial compact.


In [ ]:
activity_id = next(
    identifier
    for label, identifier in A_indices.items()
    if "transport, passenger car" in label[0].lower()
    and "gasoline" in str(label).lower()
)

lca = bc.LCA(
    demand={activity_id: 1},
    data_objs=[datapackage],
    use_distributions=False,
)
lca.lci()
lca.lcia()
lca.score


## Optional: reproduce the linear solve

This verifies the matrix orientation using the matrices already assembled by `bw2calc`.


In [ ]:
from scipy.sparse.linalg import spsolve

supply = spsolve(lca.technosphere_matrix, lca.demand_array)
inventory = lca.biosphere_matrix @ supply
manual_score = float((lca.characterization_matrix @ inventory).sum())

np.isclose(manual_score, lca.score), manual_score


## Pitfalls and extension

- Keep matrix coordinates and index files from the same export.
- Technosphere flip flags encode input signs; do not discard them.
- The one-flow method above is pedagogical, not a complete climate-change method.

## Exercise

Select a different activity by all four index-label fields and compare its `bw2calc` and direct-solve scores.


In [ ]:
exercise_matches = [
    (label, identifier)
    for label, identifier in A_indices.items()
    if "market for electricity, low voltage" in label[0].lower()
]
exercise_matches[:10]
